In [1]:
import numpy as np
import pandas as pd

In [2]:
temp_df=pd.read_csv('IMDB Dataset.csv')

In [3]:
df=temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(17)

In [8]:
df.drop_duplicates(inplace=True)

C:\Users\souha\AppData\Local\Temp\ipykernel_8940\3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
#remove tags
import re
def remove_tags(text):
    cleaned_text=re.sub(re.compile('<.*?>'),'',text)
    return cleaned_text

In [11]:
df['review']=df['review'].apply(remove_tags)

C:\Users\souha\AppData\Local\Temp\ipykernel_8940\2928534252.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(remove_tags)


In [12]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [13]:
#lowercasing
df['review']=df['review'].apply(lambda x:x.lower())

C:\Users\souha\AppData\Local\Temp\ipykernel_8940\477286236.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(lambda x:x.lower())


In [14]:
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
9995,"fun, entertaining movie about wwii german spy ...",positive
9996,give me a break. how can anyone say that this ...,negative
9997,this movie is a bad movie. but after watching ...,negative
9998,this is a movie that was probably made to ente...,negative


In [15]:
#removing stopwords
from nltk.corpus import stopwords
def remove_stopwords(text):
    new_text=[]
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [17]:
df['review']=df['review'].apply(remove_stopwords)

C:\Users\souha\AppData\Local\Temp\ipykernel_8940\1478176011.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review']=df['review'].apply(remove_stopwords)


In [18]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz e...,positive
1,wonderful little production. filming techniq...,positive
2,thought wonderful way spend time hot s...,positive
3,basically there's family little boy (jake) ...,negative
4,"petter mattei's ""love time money"" visuall...",positive
...,...,...
9995,"fun, entertaining movie wwii german spy (juli...",positive
9996,"give break. anyone say ""good hockey mo...",negative
9997,movie bad movie. watching endless series...,negative
9998,movie probably made entertain middle sc...,negative


<h2>Training the Dataset</h2>

In [22]:
X=df.iloc[:,0:1]
y=df['sentiment']

In [23]:
#label encoding the sentiment
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
y=encoder.fit_transform(y)

In [24]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [25]:
#training
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

<h3>Using BoW</h3>

In [26]:
#Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

In [27]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

<h2>Modelling</h2>

<h3>1. Gaussian Naive Bayes</h3>

In [28]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [29]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6324486730095142

<p><b>We observe a very poor accuracy of our sentiment analysis model, so we may consider switching the ML algorithm.</b></p>

<h3>2. Random Forest</h3>

In [30]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred=rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8442663995993991

<p><b>We see a much better improvement and accuracy of our sentiment analysis model.</b></p>

<h3>Applying Tf-Idf</h3>

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()

In [32]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review'])

In [33]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test,y_pred)

0.8432648973460191

<p> <b>Still we see a good accuracy of our model.</b> </p>

In [36]:
#testing
review = ["The movie was absolutely amazing. I really enjoyed it."]

# Convert the review into TF-IDF features
review_tfidf = tfidf.transform(review)

# Predict
prediction = rf.predict(review_tfidf)

# Display result
if prediction[0] == 1:
    print("Positive")
else:
    print("Negative")

Positive


In [37]:
review = ["This movie was boring and a complete waste of time."]

review_tfidf = tfidf.transform(review)

prediction = rf.predict(review_tfidf)

if prediction[0] == 1:
    print("Positive")
else:
    print("Negative")

Negative
